<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/Module%204/logo.png" width="300" alt="cognitiveclass.ai logo">
</center>


# Lab: **Building AI Agents from Scratch with Python**


Estimated time needed: **30** minutes


## Overview


This lab powers real-world applications such as a Weather AI Agent and The Daily Dish customer-service chatbot. In this hands-on project, you’ll apply core agentic AI principles to solve practical problems, design and orchestrate multi-agent systems, integrate external tools, memory, and document-based knowledge sources, and implement intelligent routing between specialized agents.

This lab is ideal for software engineers and machine learning engineers who want to understand how AI agents work under the hood and learn how to build agent-based systems from first principles using Python.


## Objectives

By the end of this lab, learners will be able to:

- Understand the core concepts of AI Agents and Multi-Agent Systems (MAS).

- Build a Weather AI Agent from scratch to retrieve and process external data using an API.

- Develop a customer-service chatbot for The Daily Dish using a PDF-based knowledge source.

- Implement agent memory to retain context and enable more personalized interactions.

- Design and integrate a multi-agent architecture where specialized agents collaborate to solve real-world problems.


## Table of Content
- [Setup and Prerequisites](#Setup-and-Prerequisites)
- [AI Agents](#AI-Agents)
- [MemoryAgent](#MemoryAgent)
- [Weather Agent](#Weather-Agent)
- [The Daily Dish Agent](#The-Daily-Dish-Agent)
- [Routing Queries](#Routing-Queries)
- [Chatbot Function: Process User Queries](#Chatbot-Function:-Process-User-Queries)


### Project Overview: 
In this hands-on project, you will build AI agents from scratch and use them to power real-world applications such as:

* A Weather AI Agent that retrieves and reports real-time weather data.

* A Daily Dish customer-service chatbot that answers questions about reservations, location, and menu from a PDF knowledge source.

**Explanation of Flow**

- User Question: The user sends a query.

- QueryRouter: Determines the correct agent to handle the question.

- WeatherAgent / DailyDishAgent: Process the question:

          - WeatherAgent fetches live API data.

          - DailyDishAgent searches the PDF knowledge base.

- MemoryAgent: Both agents can store new info or recall previous context for smarter answers.

- Response to User: The chosen agent sends back the final answer, optionally using context from MemoryAgent.


### **Setup and Prerequisites**

#### Get Your OpenWeather API Key:
Before writing any code, 
1. Go to [OpenWeather](https://home.openweathermap.org/) and **Register/Log in** to your account.  
2. Navigate to [API Keys](https://home.openweathermap.org/api_keys) and copy your **API key**, as shown in the screenshot below.


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/KlRR3k8c739PEjzS4nTUHw/openweather-api.png" >
</center>


In [ ]:
# Copy-Paste the API code here
API_KEY = #"Paste your API code here"

#### Install required packages


<span style="color:red"><b>Restart the kernel once the installation is complete.</b></span>


In [ ]:
!pip install nltk PyPDF2 scikit-learn requests numpy

In [ ]:
import re
import requests
import nltk
import numpy as np
from PyPDF2 import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt")

In [ ]:
!wget 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf'
faq_pdf_path = "The-Daily-Dish-FAQ.pdf"

# AI Agents 
In this lab, we create three specialized agents, each responsible for a distinct role within the system:

**Memory Agent** – Stores and recalls information from previous interactions, enabling more personalized and consistent responses across agents.

**Weather Agent** – Fetches real-time weather information from an external API and uses memory to provide context-aware responses.

**Daily Dish Agent** – Acts as a customer-service agent that answers restaurant-related questions using a PDF-based knowledge source.

These three agents work together as a multi-agent system, coordinated through routing logic to handle user queries efficiently and intelligently.


# MemoryAgent
The MemoryAgent is a simple storage component that allows your agents to remember things during a conversation or session. Think of it like a notepad for your AI agents — they can write down key information and refer back to it later.

**Purpose:**

- Stores information as key-value pairs.

- Lets other agents recall previous responses or context, which is essential for multi-step reasoning or maintaining conversation state.

**How it works:**

- store(key, value) — saves information under a unique key.

- recall(key) — retrieves the information associated with that key, if it exists.

**Why we define it first:**

- Both WeatherAgent and DailyDishAgent can use MemoryAgent.

- Introducing it first helps to understand how agents share memory and maintain context.


In [ ]:
class MemoryAgent:
    def __init__(self):
        # Initialize an empty dictionary to store agent memory
        # This acts as shared context across multiple agents
        self.memory = {}

    def store(self, key, value):
        """
        Store information in memory using a key-value pair.
        Example:
        key   -> 'last_weather_query'
        value -> 'Weather in Bangalore is 28°C'
        """
        self.memory[key] = value

    def recall(self, key=None):
        """
        Retrieve information from memory.
        
        - If a specific key is provided, return the stored value for that key.
        - If no key is provided, return the entire memory dictionary.
        
        This allows agents to:
        - Maintain conversation context
        - Avoid repeating work
        - Adapt responses based on past interactions
        """
        if key:
            return self.memory.get(key)
        return self.memory


# Weather Agent
The WeatherAgent is a task-specific AI agent responsible for retrieving real-time weather information for a given city.

**Perception:**
The agent receives a city name as input.

**Action (Tool Use):**
It calls the OpenWeather API to fetch live weather data such as temperature and conditions.

**Memory:**
The agent uses a shared MemoryAgent to store previously retrieved weather data for each city.

**Reasoning:**
If the agent has past weather data in memory, it compares the previous temperature with the current one and includes this context in its response.


In [ ]:
class WeatherAgent:
    # Initialize the Weather Agent
    # - api_key: OpenWeather API key
    # - memory_agent: shared memory agent to store past weather data
    def __init__(self, api_key, memory_agent):
        self.api_key = api_key              # Store the API key
        self.memory = memory_agent          # Reference to the memory agent
        self.url = "http://api.openweathermap.org/data/2.5/weather"  # Weather API endpoint

    # Generate a weather response for a given city
    def answer(self, city):
        # Prepare query parameters for the API call
        params = {
            "q": city,                      # City name
            "appid": self.api_key,          # API authentication key
            "units": "metric"               # Return temperature in Celsius
        }

        # Call the OpenWeather API
        res = requests.get(self.url, params=params)

        # Handle API errors or failed requests
        if res.status_code != 200:
            return "I couldn’t retrieve the weather right now."

        # Parse the JSON response
        data = res.json()

        # Retrieve previous weather data for this city from memory (if available)
        previous = self.memory.recall(city)

        # Store the current weather data in memory for future context
        self.memory.store(city, data["main"])

        # Construct a natural-language weather response
        response = (
            f"The current weather in {city} is {data['weather'][0]['description']} "
            f"with a temperature of {data['main']['temp']}°C."
        )

        # If previous data exists, add contextual comparison
        if previous:
            response += f" Earlier it was {previous['temp']}°C."

        # Return the final response to the user
        return response


# The Daily Dish Agent

Here's your challenge: You've been hired by "The Daily Dish," a popular restaurant with a customer service problem. Chef Maria, the owner, needs your help: "My team spends hours answering the same questions about reservations and menus," she explains. "Build me a chatbot that feels personal, to handle these inquiries so my staff can focus on creating exceptional dining experiences." Chef Maria sends you a PDF of frequently asked questions (FAQs) and now you must get the job done!

The Daily Dish Agent is a customer-service AI agent designed to answer questions about The Daily Dish restaurant using information stored in a PDF document.

This agent:

- Loads and processes the restaurant’s FAQ PDF as its knowledge source.

- Converts questions from the document into numerical vectors using TF-IDF, enabling semantic similarity matching.

- Compares a user’s question with the stored FAQ questions using cosine similarity.

- Retrieves and returns the most relevant answer when the similarity score exceeds a defined threshold.

Unlike traditional rule-based chatbots, the Daily Dish Agent does not rely on hard-coded responses. Instead, it uses information retrieval techniques, allowing it to respond flexibly to different phrasings of the same question.

**Steps in the Daily Dish Agent**

- Load the The Daily Dish FAQ PDF and extract all text content.

- Clean and preprocess the extracted text for consistency.

- Parse the text into structured question–answer pairs.

- Convert FAQ questions into numerical vectors using TF-IDF to build the agent’s knowledge base.

- Receive and preprocess a user query, then compute similarity against stored FAQ questions.

- Identify and return the most relevant FAQ answer or a fallback response if confidence is low.



#### Load the FAQ PDF and Extract text


In [ ]:
# Path to the Daily Dish FAQ PDF file
FAQ_PDF = "The-Daily-Dish-FAQ.pdf"
def load_faq_pdf(pdf_path):
    # Create a PDF reader object
    reader = PdfReader(pdf_path)
    
    # Initialize an empty string to store extracted text
    text = ""
    
    # Loop through each page in the PDF
    for page in reader.pages:
        # Extract text from the page and append it
        text += page.extract_text()
    
    # Return the complete extracted text
    return text

# Extract all text content from the FAQ PDF
faq_text = load_faq_pdf(FAQ_PDF)


#### Clean and normalize the extracted text


In [ ]:
def clean_text(text):
    # Replace multiple spaces, newlines, or tabs with a single space
    cleaned_text = re.sub(r"\s+", " ", text)
    
    # Remove leading and trailing whitespace
    return cleaned_text.strip()

#### Parse the FAQ into structured question–answer pairs


In [ ]:
# Function to parse a FAQ text into structured question-answer pairs
def parse_faq(text):
    # Initialize an empty list to store the parsed Q&A pairs
    faq_pairs = []

    # Define a regex pattern to match "Q: <question> A: <answer>" blocks
    # - (.*?) captures the question text lazily
    # - (.*?)(?=\n\s*\d+\.\s*Q:|\Z) captures the answer lazily until the next numbered question or end of text
    pattern = r"Q:\s*(.*?)\s*A:\s*(.*?)(?=\n\s*\d+\.\s*Q:|\Z)"

    # Find all matches of the pattern in the text
    # re.DOTALL allows '.' to match newline characters so multi-line answers are captured
    matches = re.findall(pattern, text, re.DOTALL)

    # Loop over each matched question-answer tuple
    for q, a in matches:
        # Append a dictionary with cleaned question and answer to the list
        # - clean_text() removes extra spaces and normalizes the text
        # - question is converted to lowercase for consistent indexing/search
        faq_pairs.append({
            "question": clean_text(q.lower()),
            "answer": clean_text(a)
        })
    
    # Return the list of structured Q&A dictionaries
    return faq_pairs

# Parse the FAQ text into structured data
faq_data = parse_faq(faq_text)

# Extract just the list of questions from the structured data
faq_questions = [item["question"] for item in faq_data]

# Extract just the list of answers from the structured data
faq_answers = [item["answer"] for item in faq_data]


#### Preprocess user queries


In [ ]:
# Class to preprocess and enhance user queries
class QueryProcessor:
    def process(self, query):
        # Convert the entire query to lowercase for consistency
        query = query.lower()

        # Remove all punctuation and special characters, keeping only letters, numbers, and spaces
        query = re.sub(r"[^\w\s]", "", query)

        # Define a dictionary of synonyms to expand the query
        # Keys are words to look for in the query
        # Values are additional terms to append if the key is found
        synonyms = {
            "location": "located address",
            "where": "located address",
            "reservation": "reserve booking",
            "menu": "food dishes",
            "fish": "seafood"
        }

        # Loop through the synonyms dictionary
        for k, v in synonyms.items():
            # If the keyword exists in the query, append its corresponding synonym
            # This helps improve matching or search results
            if k in query:
                query += " " + v

        # Return the processed and enhanced query
        return query


#### Vectorize FAQ questions using TF-IDF and Retrieve the most relevant answer


In [ ]:
# Agent class for handling FAQ-style question-answer interactions
class DailyDishAgent:
    def __init__(self, questions, answers):
        # Store the list of answers corresponding to the FAQ questions
        self.answers = answers

        # Initialize a TF-IDF vectorizer for converting text into numerical vectors
        # - stop_words="english" removes common English words (like "the", "is") for better focus on keywords
        # - ngram_range=(1, 2) considers both single words (unigrams) and pairs of words (bigrams) for richer text representation
        self.vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2)
        )

        # Fit the vectorizer to the questions and transform them into TF-IDF vectors
        # This creates a matrix where each row represents a question and columns represent weighted terms
        self.doc_vectors = self.vectorizer.fit_transform(questions)

      # Method to find the most relevant answer for a given user query
    def answer(self, query):
        # Transform the user query into a TF-IDF vector using the same vectorizer as the questions
        query_vector = self.vectorizer.transform([query])
    
        # Compute cosine similarity between the query vector and all stored question vectors
        # - cosine similarity measures how close the query is to each question
        # - similarities[0] gives a 1D array of similarity scores
        similarities = cosine_similarity(query_vector, self.doc_vectors)[0]
    
        # Find the index of the question with the highest similarity score
        best_idx = np.argmax(similarities)
    
        # If the highest similarity is below a threshold (0.08), consider it "no match"
        # - This prevents returning irrelevant answers
        if similarities[best_idx] < 0.08:
            return None
    
        # Return the answer corresponding to the most similar question
        return self.answers[best_idx]



# Routing Queries

In a multi-agent system, queries are handled by specialized agents, such as a WeatherAgent for weather questions and a DailyDishAgent for restaurant FAQs.

The route_query function acts as a traffic controller: it checks the user’s input for keywords (like "weather", "rain", "forecast") and routes the query accordingly. If a keyword matches, the query goes to the WeatherAgent; otherwise, it defaults to the DailyDishAgent.

This ensures queries are processed by the most suitable agent, improving relevance. Preprocessing with a QueryProcessor and adding fallback responses can further enhance accuracy.


In [ ]:
# Function to route a user query to the appropriate agent
def route_query(query):
    # Define keywords that indicate the query is related to weather
    weather_keywords = [
        "weather", "rain", "raining", "forecast",
        "temperature", "hot", "cold", "humidity"
    ]

    # Convert the query to lowercase for case-insensitive matching
    query = query.lower()

    # Check if any weather-related keyword exists in the query
    for word in weather_keywords:
        if word in query:
            # Route to the weather agent
            return "weather"

    # If no weather keywords are found, route to the DailyDish agent
    return "daily_dish"


#### Initialize Agents and API Keys


In [ ]:
# Initialize the query processor to clean and expand user queries
query_processor = QueryProcessor()

# Initialize a memory agent to store and recall context if needed
memory_agent = MemoryAgent()

# 👉 Replace with your real API key for the weather API
WEATHER_API_KEY = API_KEY

# Initialize the WeatherAgent with API key and memory
weather_agent = WeatherAgent(WEATHER_API_KEY, memory_agent)

# Initialize the DailyDishAgent with FAQ questions and answers
daily_dish_agent = DailyDishAgent(faq_questions, faq_answers)

# Define the city for weather queries
RESTAURANT_CITY = "New york"


# Chatbot Function: Process User Queries


In [ ]:
def chatbot(user_question):
    # Determine which agent should handle the query (Weather or DailyDish)
    route = route_query(user_question)

    # Preprocess the query (lowercase, clean text, expand synonyms)
    processed = query_processor.process(user_question)

    # If the query is weather-related, get answer from WeatherAgent
    if route == "weather":
        return weather_agent.answer(RESTAURANT_CITY)

    # Otherwise, get answer from DailyDishAgent
    answer = daily_dish_agent.answer(processed)
    if answer:
        return answer

    # Fallback message for unrecognized queries
    return "I’m not sure about that. Please ask a question related to The Daily Dish."


#### Interactive Chat Loop


In [ ]:
print("🍽️ Welcome to The Daily Dish Chatbot!")
print("Type 'exit' to end the conversation.\n")
while True:
    user_input = input("You: ")

    # Exit conditions for ending the chat
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Chatbot: 👋 Thanks for visiting The Daily Dish!")
        break

    # Process the query and print the chatbot's response
    print("Chatbot:", chatbot(user_input))
    print()



# Congratulations! 
You've successfully gained hands-on experience in building and integrating AI agents. You learned the core concepts of AI Agents and Multi-Agent Systems (MAS), created a Weather AI Agent to retrieve and process real-world data, and developed a customer-service chatbot for The Daily Dish using a PDF-based knowledge source.


## Authors

Sowmyaa 

Sathya Priya


<div align="center">
© Copyright IBM Corporation. All rights reserved.
</div>
